# Problem 91

The points $P(x_1, y_1)$ and $Q(x_2, y_2)$ are plotted at integer co-ordinates and are joined to the origin, $O(0,0)$, to form $\triangle OPQ$.

<div class="center">
<img src="resources/images/0091_1.png?1678992052" class="dark_img" alt=""><br></div>

There are exactly fourteen triangles containing a right angle that can be formed when each co-ordinate lies between $0$ and $2$ inclusive; that is, $0 \le x_1, y_1, x_2, y_2 \le 2$.

<div class="center">
<img src="resources/images/0091_2.png?1678992052" alt=""><br></div>

Given that $0 \le x_1, y_1, x_2, y_2 \le 50$, how many right triangles can be formed?

In [13]:
# dynamic programming problem
M = 50
x = M
y = M
# right angle with coordinate system + right angles at 45 degrees + others
3 * x * y

7500

In [ ]:
from math import gcd

M = 50
total = 3 * M * M # triangles on axes

for x in range(1, M+1):
    for y in range(1, x+1):
        g = gcd(x,y)
        p, q = x // g, y // g
        k1 = min((M - x) // q, y // p) # other point is (x-tq, y+tp), between 0 and M
        k2 = min(x // q, (M - y) // p) # other direction with s
        s = k1 + k2
        if x == y:
            total += s
        else:
            total += 2*s # reflect across diagonal
print(total)

14234


# Problem 92

A number chain is created by continuously adding the square of the digits in a number to form a new number until it has been seen before.
For example,
$$\begin{align*}
&44 \to 32 \to 13 \to 10 \to \mathbf 1 \to \mathbf 1\\
&85 \to \mathbf{89} \to 145 \to 42 \to 20 \to 4 \to 16 \to 37 \to 58 \to \mathbf{89}
\end{align*}$$
Therefore any chain that arrives at $1$ or $89$ will become stuck in an endless loop. What is most amazing is that EVERY starting number will eventually arrive at $1$ or $89$.
How many starting numbers below ten million will arrive at $89$?


In [ ]:
from itertools import combinations_with_replacement
from math import factorial

def count_arrangements(combo):
    counts = [combo.count(d) for d in range(10)]
    result = factorial(len(combo))
    for c in counts:
        result //= factorial(c)
    return result

sq = {d: d*d for d in range(10)}
def digit_sq_sum(n):
    return sum(sq[int(c)] for c in str(n))

ends = {1: 1, 89: 89}
for n in range(1, 649):
    seq = [n]
    curr = seq[0]
    while curr not in ends:
        curr = digit_sq_sum(curr)
        seq.append(curr)
    end = ends[curr]
    for num in seq:
        ends[num] = end

count = 0
for combo in combinations_with_replacement(range(10), 7):
    if combo == (0,)*7:
        continue
    dss = sum(sq[d] for d in combo)
    if ends[dss] == 89:
        count += count_arrangements(combo)
print(count)

8581146


# Problem 93

By using each of the digits from the set, $\{1, 2, 3, 4\}$, exactly once, and making use of the four arithmetic operations ($+, -, \times, /$) and brackets/parentheses, it is possible to form different positive integer targets.
For example,
$$\begin{align*}
8 &= (4 \times (1 + 3)) / 2\\
14 &= 4 \times (3 + 1 / 2)\\
19 &= 4 \times (2 + 3) - 1\\
36 &= 3 \times 4 \times (2 + 1)
\end{align*}$$
Note that concatenations of the digits, like $12 + 34$, are not allowed.
Using the set, $\{1, 2, 3, 4\}$, it is possible to obtain thirty-one different target numbers of which $36$ is the maximum, and each of the numbers $1$ to $28$ can be obtained before encountering the first non-expressible number.
Find the set of four distinct digits, $a \lt b \lt c \lt d$, for which the longest set of consecutive positive integers, $1$ to $n$, can be obtained, giving your answer as a string: <i>abcd</i>.


In [ ]:
from itertools import combinations
from fractions import Fraction  # or think about why this avoids float drift

def reachable(numbers):
    """
    numbers: a tuple/frozenset of raw digits, e.g. (1,2,3,4)
    returns: a set of all values reachable by combining them
             with +,-,*,/ and any parenthesization
    """
    numbers = tuple(numbers)
    n = len(numbers)
    
    # --- base case ---
    if n == 1:
        return { numbers[0] }   # what's reachable from a single number?
    
    results = set()
    
    # --- recursive case: try every way to split into left/right ---
    for size in range(1, n):  # left-group sizes to try
        for left_idx in combinations(range(n), size):
            left_nums  = tuple(numbers[i] for i in left_idx)
            right_nums = tuple(numbers[i] for i in range(n) if i not in left_idx)
            
            left_results  = reachable(left_nums)   # recursive call
            right_results = reachable(right_nums)  # recursive call
            
            for a in left_results:
                for b in right_results:
                    # try +, -, *, / (both orders where relevant)
                    # add valid results to `results`
                    results.add(a + b)
                    results.add(a - b)
                    results.add(b - a)
                    results.add(a * b)
                    if b != 0:
                        results.add(Fraction(a, b))
                    if a != 0:
                        results.add(Fraction(b, a))

    return results

def longest_run(results):
    n = 1
    while n in results:
        n += 1
    return n - 1

max_run = 0
max_set = ()
for i in range(0, 10):
    for j in range(i+1, 10):
        for k in range(j+1, 10):
            for l in range(k+1, 10):
                nums = reachable((i,j,k,l))
                nums = {int(n) for n in nums if n > 0 and n.denominator == 1}
                if longest_run(nums) > max_run:
                    max_run = longest_run(nums)
                    max_set = (i,j,k,l)
print(max_set)

(1, 2, 5, 8)


# Problem 94

It is easily proved that no equilateral triangle exists with integral length sides and integral area. However, the <dfn>almost equilateral triangle</dfn> $5$-$5$-$6$ has an area of $12$ square units.
We shall define an <dfn>almost equilateral triangle</dfn> to be a triangle for which two sides are equal and the third differs by no more than one unit.
Find the sum of the perimeters of all <dfn>almost equilateral triangles</dfn> with integral side lengths and area and whose perimeters do not exceed one billion ($1\,000\,000\,000$).


In [ ]:
# Pell equation

def pell_sols(limit_u):
    # start at smallest nontrivial solution to u^2 - 3r^2 = 4, with u = 3i+-4
    u, r = 4, 2

    while u <= limit_u:
        yield u, r

        # recurrence: multiply (u + r*sqrt(3)) by fundamental unit 2 + sqrt(3)
        # gives (u, r) -> (2u + 3r, u + 2r)
        u, r = 2*u + 3*r, u + 2*r


def sum_of_perimeters(perimeter_limit):
    total = 0

    for u, r in pell_sols(perimeter_limit):
        # minus case - sides i and i-1
        if (u + 4) % 3 == 0:
            i = (u + 4) // 3
            equal_side = i - 1

            if (i * r) % 4 == 0: # area is int
                perimeter = i + 2 * equal_side
                if perimeter <= perimeter_limit:
                    #print(i, equal_side, equal_side)
                    total += perimeter


        # plus case - sides i and i+1
        if (u - 4) % 3 == 0:
            i = (u - 4) // 3
            equal_side = i + 1

            if (i * r) % 4 == 0:
                perimeter = i + 2 * equal_side
                if perimeter <= perimeter_limit:
                    #print(i, equal_side, equal_side)
                    total += perimeter

    return total - 2 # eliminate 0 1 1 case

sum_of_perimeters(1_000_000_000)

518408346

# Problem 95
The proper divisors of a number are all the divisors excluding the number itself. For example, the proper divisors of $28$ are $1$, $2$, $4$, $7$, and $14$. As the sum of these divisors is equal to $28$, we call it a perfect number.
Interestingly the sum of the proper divisors of $220$ is $284$ and the sum of the proper divisors of $284$ is $220$, forming a chain of two numbers. For this reason, $220$ and $284$ are called an amicable pair.
Perhaps less well known are longer chains. For example, starting with $12496$, we form a chain of five numbers:
$$12496 \to 14288 \to 15472 \to 14536 \to 14264 (\to 12496 \to \cdots)$$
Since this chain returns to its starting point, it is called an amicable chain.
Find the smallest member of the longest amicable chain with no element exceeding one million.


In [2]:
# Source - https://stackoverflow.com/a/70638665
# Posted by Jérôme Richard
# Retrieved 2026-09-06, License - CC BY-SA 4.0

import numba as nb

@nb.njit('List(int_)(int_)')
def get_prime_divisors(n):
    divisors = []
    while n % 2 == 0:
        divisors.append(2)
        n //= 2
    while n % 3 == 0:
        divisors.append(3)
        n //= 3
    i = 5
    while i*i <= n:
        for k in (i, i+2):
            while n % k == 0:
                divisors.append(k)
                n //= k
        i += 6
    if n > 1:
        divisors.append(n)
    return divisors

@nb.njit('List(int_)(int_)')
def get_divisors(n):
    divisors = []
    if n == 1:
        divisors.append(1)
    elif n > 1:
        prime_factors = get_prime_divisors(n)
        divisors = [1]
        last_prime = 0
        factor = 0
        slice_len = 0
        # Find all the products that are divisors of n
        for prime in prime_factors:
            if last_prime != prime:
                slice_len = len(divisors)
                factor = prime
            else:
                factor *= prime
            for i in range(slice_len):
                divisors.append(divisors[i] * factor)
            last_prime = prime
        divisors.sort()
    return divisors


In [22]:
# similar to problem 92, we should use a dictionary to map values for quicker lookup
# all numbers in the same chain have the same chain length
# must keep track of if hits 1 million, or if a subsequence repeats, that is not amicable

from functools import lru_cache

@lru_cache(maxsize=None)
def divisor_sum(n):
    return sum(get_divisors(n)[:-1]) # exclude n itself

chain_lengths = {}

for i in range(1, 1_000_000):
    if i in chain_lengths: # already calculated
        continue

    seen = {} # use dict for O(1) lookup, to find duplicates
    chain = []
    curr = i
    while curr not in seen and curr <= 1_000_000:
        seen[curr] = len(chain)
        chain.append(curr)
        curr = divisor_sum(curr)
    if curr > 1_000_000:
        cycle, length = chain, 0
    else:
        start = seen[curr]
        cycle = chain[start:]
        length = len(cycle)

    for num in cycle:
        chain_lengths[num] = length


max_len = max(chain_lengths.values())
smallest = min(num for num, length in chain_lengths.items() if length == max_len)
print(smallest)#, max_len)

14316
